# Chapter 13: Computer Vision
**Part IV — Specialist Domains**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

Image classification, object detection (YOLO, Faster R-CNN), segmentation, and CLIP.

## Learning Objectives
See Chapter 13 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Using a pre-trained diffusion model with Diffusers


In [ ]:
# Using a pre-trained diffusion model with Diffusers
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

prompt = "A futuristic city at sunset, digital art, 4K"
negative_prompt = "blurry, low quality, distorted"

image = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
).images[0]

image.save("output.png")

## Transfer learning: fine-tune ResNet-18 on a small dataset


In [ ]:
# Transfer learning: fine-tune ResNet-18 on a small dataset
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import FakeData

# Simulate a small 4-class dataset (e.g. medical imaging categories)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = FakeData(size=200, image_size=(3, 224, 224), num_classes=4, transform=transform)
val_ds   = FakeData(size=50,  image_size=(3, 224, 224), num_classes=4, transform=transform)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16)

# Load pretrained ResNet-18 and replace final layer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all layers except the final fully-connected layer
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 4)   # 4 output classes
model = model.to(device)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params (fc layer only): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print()

# Quick training run (2 epochs to demonstrate)
for epoch in range(2):
    model.train(); train_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward(); optimizer.step()
        train_loss += loss.item()
    
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for X, y in val_loader:
            preds = model(X.to(device)).argmax(1).cpu()
            correct += (preds == y).sum().item(); total += len(y)
    
    print(f"Epoch {epoch+1}: loss={train_loss/len(train_loader):.4f}, val_acc={correct/total:.4f}")

print("\nTransfer learning: only the final layer was trained — 99.9% of parameters were frozen.")

---

## 📚 Review Questions

See Chapter 13 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 13 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*